# Lesson 11 作業｜先問再下載、批次縮小、圖片索引

把每一格的 `None` 或 `# TODO` 補完，再按 Run Cell。每題最後的 `assert` 通過、印出「完成」，就代表答對了。
只用本機練習站與 `fixtures/gallery`，不需要外部網路。


### Notebook 使用方式

先選擇課程 Python 環境，再由上到下執行每一格（Shift + Enter）。先執行工作資料夾設定及前面的匯入。
若看到 `NameError`，先確認變數名稱，並回到上方執行建立該變數的格子；必要時重新啟動核心並從頭執行。練習或作業格要先完成 TODO，再執行檢查。


## 設定工作資料夾

請先執行下一格。使用 VS Code 開啟完整教材資料夾；本堂的 `fixtures/` 和 `output/` 會以本堂資料夾為基準。


In [ ]:
import os
from pathlib import Path

# 從教材根目錄或本堂資料夾啟動 Notebook 都可以。
current = Path.cwd().resolve()
lesson_name = 'lesson11_images_and_files'
candidates = [current, *current.parents]
lesson_path = next((p for p in candidates if p.name == lesson_name and (p / 'fixtures').is_dir()), None)
if lesson_path is None:
    lesson_path = next((p / 'lessons' / lesson_name for p in candidates if (p / 'lessons' / lesson_name / 'fixtures').is_dir()), None)
if lesson_path is None:
    raise FileNotFoundError('找不到本堂教材。請用 VS Code 開啟完整教材資料夾，重新啟動核心後再執行。')
os.chdir(lesson_path)
print('工作資料夾：', Path.cwd())

import sys
tools_path = str(lesson_path.parents[1] / 'tools')
if tools_path not in sys.path:
    sys.path.insert(0, tools_path)


In [ ]:
import io
import re
import shutil
import time
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from PIL import Image, ImageOps

from practice_site import start_server, stop_server

server, BASE_URL = start_server()  # 本機練習站：/gallery 是圖庫，/images/ 放圖片
GALLERY = Path("fixtures/gallery")  # 同一批圖片的本機副本，第 7 題起會用到
print("本機練習站：", BASE_URL)


## 已經寫好的小工具

這兩個函式在練習裡寫過了，這裡直接給你用。`image_urls` 是圖庫頁上 17 個 `<img>` 的完整網址。


In [ ]:
def filename_from_url(url):
    return Path(urlparse(url).path).name


def download_image(url, folder):
    response = requests.get(url, timeout=10)
    if response.status_code != 200 or not response.headers.get("Content-Type", "").startswith("image/"):
        return None
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / filename_from_url(url)
    path.write_bytes(response.content)
    return path


page_url = f"{BASE_URL}/gallery"
soup = BeautifulSoup(requests.get(page_url, timeout=10).text, "html.parser")
image_urls = [urljoin(page_url, img.get("data-src") or img["src"]) for img in soup.select("img")]
print(len(image_urls), "個圖片網址")


## 第 1 題｜先問大小，再決定要不要下載

大檔案下載前，可以先用 `requests.head(url)` 只拿標頭（不下載內容），看大小再決定。
寫 `download_small_images(urls, folder, max_bytes=40_000)`：對每個網址先送 HEAD 請求，
**三個條件都符合**才用 `download_image` 下載：

- 狀態碼是 200
- `Content-Type` 以 `image/` 開頭
- `Content-Length`（轉成整數）小於 `max_bytes`

回傳「實際下載的檔名」list，依網址順序。


In [ ]:
def download_small_images(urls, folder, max_bytes=40_000):
    # TODO: 請在這裡寫程式
    return None


In [ ]:
hw1_folder = Path("output/hw1")
shutil.rmtree(hw1_folder, ignore_errors=True)

names = download_small_images(image_urls, hw1_folder)
expected = [
    "cover_01.jpg", "cover_02.jpg", "cover_03.jpg", "cover_04.jpg", "cover_05.jpg",
    "cover_08.jpg", "cover_10.jpg", "cover_12.jpg", "banner.png", "landscape.jpg", "icon.png",
]
assert names == expected, names
assert sorted(path.name for path in hw1_folder.iterdir()) == sorted(expected)
assert all(path.stat().st_size < 40_000 for path in hw1_folder.iterdir())
print("第 1 題完成：下載了", len(names), "張，略過太大的與有問題的")


## 第 2 題｜批次縮小：最大寬度 600

寫 `shrink_all(source_folder, target_folder, max_width=600)`：
把 `source_folder` 裡所有 `.jpg`、`.png` 圖片轉成 JPEG（`quality=85`），存到 `target_folder`，檔名副檔名換成 `.jpg`。

- 寬度超過 `max_width` 的，依比例縮到寬 `max_width`；沒有超過的**不要放大**
- 有透明的 PNG（RGBA）要先貼到白色背景上
- 回傳 dict：新檔名 → 新的尺寸 `(寬, 高)`

提示：`picture.thumbnail((max_width, 10_000))` 會在原圖上依比例縮小，而且不會放大。


In [ ]:
def shrink_all(source_folder, target_folder, max_width=600):
    # TODO: 請在這裡寫程式
    return None


In [ ]:
hw2_folder = Path("output/hw2")
shutil.rmtree(hw2_folder, ignore_errors=True)

sizes = shrink_all(GALLERY, hw2_folder)
assert len(sizes) == 15
assert sizes["landscape.jpg"] == (600, 338), sizes["landscape.jpg"]
assert sizes["banner.jpg"] == (600, 200), sizes["banner.jpg"]
assert sizes["cover_01.jpg"] == (318, 395), "沒超過 600 寬的不要放大"
assert sizes["icon.jpg"] == (64, 64)
for name, size in sizes.items():
    with Image.open(hw2_folder / name) as saved:
        assert (saved.format, saved.mode, saved.size) == ("JPEG", "RGB", size), name
print("第 2 題完成：", {name: size for name, size in sizes.items() if name in {"landscape.jpg", "banner.jpg"}})


## 第 3 題｜圖片索引表

用 pandas 讀 `GALLERY / "manifest.csv"`（欄位 `file`、`title`、`rating`、`kind`，編碼 `utf-8-sig`），
再用 Pillow 為每張圖片補上三欄，存成 `index_df`：

- `width`、`height`：圖片的寬與高
- `kb`：檔案大小（位元組 ÷ 1024），四捨五入到小數 1 位

欄位順序是 `file, title, rating, kind, width, height, kb`。再存成 `output/hw3/index.csv`（`encoding="utf-8-sig"`、不要 index）。


In [ ]:
hw3_path = Path("output/hw3/index.csv")

# TODO: 請在這裡寫程式
index_df = None


In [ ]:
assert index_df.shape == (15, 7)
assert list(index_df.columns) == ["file", "title", "rating", "kind", "width", "height", "kb"]
assert index_df.loc[index_df["file"] == "cover_11.jpg", "kb"].iloc[0] == 62.5
assert index_df.sort_values("kb").iloc[-1]["file"] == "cover_11.jpg"
assert index_df.loc[index_df["kind"] == "book", "width"].mean() == 260.5
assert index_df["kb"].sum() == round(index_df["kb"].sum(), 1)
saved = pd.read_csv(hw3_path, encoding="utf-8-sig")
assert saved.shape == (15, 7) and list(saved["file"]) == list(index_df["file"])
print("第 3 題完成：最大的圖是", index_df.sort_values("kb").iloc[-1]["file"])


## 收尾：關閉練習站


In [ ]:
stop_server(server)
print("練習站已關閉")
